# Plot and evaluate summed/ extrapolated scores: Erosion

In [ ]:
## IMPORTS

%load_ext autoreload
%autoreload 2

import os
import sys

from torchvision import transforms
from torch.utils.data import DataLoader
import torch


import os
import numpy as np
from torch.utils.data import Dataset

import matplotlib.pyplot as plt
import random
from pathlib import Path 


# from ra_utils.autoscora.autoscorRA_Pipeline.scoring.src.io_scoring_method import io_scoring
# from ra_utils.autoscora.autoscorRA_Pipeline.scoring.src.run_utils import (
#     paths_list_scores_list_from_score_types,
#     restructure_paths_and_scores,
#     restructure_paths_and_scores_v2
# )

import pandas as pd
from typing import List 

import ra_utils
import ra_utils.utils.config_parser
import ra_utils.networks.loss_function


import ra_utils.data.dataloader_CR_patches
from ra_utils.data.dataloader_CR_patches import (
    load_img_SHS_patch_data,
    df_scores_to_dct_list,
)

import ra_utils.data.dataloader_CR_patches
from monai.data import Dataset, DataLoader
from monai.transforms import (
    Compose
)
import torch
from torch.utils.data import WeightedRandomSampler, DataLoader


from ra_utils.data.dataloader_CR_patches import (
    load_img_SHS_patch_data,
    dataset_and_loader,
    dataset_and_loader_several,
    df_scores_to_dct_list,
    make_paths_dataframe,
    restructure_paths_and_scores,
    restructure_paths_and_scores_v2,
    exclude_ROIS_according_surgery_status,
    split_training_val_test__on_patient_level,
    process_several_score_groups,
    process_single_score_group,
    load_img_SHS_patch_data
)

import yaml
from importlib import resources


import ra_utils.networks.architecture
from ra_utils.networks.architecture import (
    ResNet18Encoder,
    ResNet34Encoder,
    ResNet50Encoder,
    make_mlp,
    EncoderClassifierNetwork,
    MultiModalImageScoreTypeNetwork,
    ROI_type_encoder,
    model_interface_forward
)
import numpy as np
from ra_utils.autoscora.autoscorRA_Pipeline.scoring.src.network import Custom_VGG
from ra_utils.utils.utils_SHS_scoring import get_classes
import ra_utils.utils.utils
import torch.nn as nn


import ra_utils.mtan.im2im_pred.model_resnet_mtan.resnet_recon_mtan

from ra_utils.mtan.im2im_pred.model_resnet_mtan.resnet_recon_mtan import (
    #MTANResNetRecon
    MTANReconCls,
    build_mtan_recon_cls
)


from ra_utils.data.data_utils import (
    extract_extras_from_filename
)


from ra_utils.training.scores_SHS.model_builders import build_models_AE
from tqdm.notebook import tqdm

from ra_utils.progressionlearning.models.builder import (
    build_MTANAE
)
from ra_utils.progressionlearning.models.MTANUNet import (
    MTANRecUnet, 
    MTANRecUnet_v2,
    MTANRecUnet_v3
)
import monai
from monai.networks.nets import BasicUNet, UNet


from ra_utils.progressionlearning.models.builder import (
    build_MTANAE, 
    build_MTANAE_v2
)
from ra_utils.progressionlearning.models.MTANUNet import (
    MTANRecUnet,
    MTANRecUnet_v2
)
import monai
import monai.networks.nets
from monai.networks.nets import BasicUNet, UNet
from typing import Dict, List

import ra_utils.mtan.im2im_pred.model_resnet_mtan.resnet_mtan
from ra_utils.training.scores_SHS.model_builders import build_models_AE_v2, build_models_AE_v1_and2


# wrap model: 

from ra_utils.networks.architecture import (
    MultiModalImageScoreTypeNetworkAE,
    ROI_type_encoder
)



# get all scores types / roi types

import pingouin as pg


from ra_utils.data.shap_sums import (
    add_JSN_ERO_sums, 
    double_scoring_make_merge_id,
    limit_treatment_number
)


from ra_utils.data.icc import compute_icc3

from ra_utils.training.scores_SHS.scores_SHS_training_lib import (
    calculate_some_classification_metrics
)
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    balanced_accuracy_score,
)


import os
import mlflow
from mlflow.tracking import MlflowClient
import json
import yaml
from sklearn.metrics import balanced_accuracy_score

from pprint import pprint

# CODE: 
from ra_utils.evaluation.single_SHS import (
    combine_predictions, 
    get_main_metrics
)


from ra_utils.data.shap_sums import (
   #max_possible_score, 
    sum_and_extrapolate_scores_df,
    generate_score_differences,
    sum_and_extrapolate_scores_df_ERO_H,
    sum_and_extrapolate_scores_df_ERO_F
    #sum_scores_df
)



import ra_utils.visualization.plot_SHS_scores
from ra_utils.visualization.plot_SHS_scores import (
    plot_SHS_deltas, 
    plot_SHS_sums
)


with resources.files("ra_utils.resources.scores_metadata").joinpath("roi_scores_matching.csv") as f:
    df_scores_meta = pd.read_csv(f)
H_ERO_scores = sorted(df_scores_meta[(df_scores_meta["ERO_or_JSN"] == "ERO") & (df_scores_meta["region"] == "H")]["score_name"].unique())
F_ERO_scores = sorted(df_scores_meta[(df_scores_meta["ERO_or_JSN"] == "ERO") & (df_scores_meta["region"] == "F")]["score_name"].unique())
H_JSN_scores = sorted(df_scores_meta[(df_scores_meta["ERO_or_JSN"] == "JSN") & (df_scores_meta["region"] == "H")]["score_name"].unique())
F_JSN_scores = sorted(df_scores_meta[(df_scores_meta["ERO_or_JSN"] == "JSN") & (df_scores_meta["region"] == "F")]["score_name"].unique())


In [ ]:
mlflow_runs_dir = "/home/cwatzenboeck/data/mlflow_cirpc_tmp/RA/data/"
os.environ["MLFLOW_TRACKING_URI"] = mlflow_runs_dir

client = MlflowClient()


In [ ]:
## Input: 

runs_to_examine = {
    "H_ERO_01": {
        #"Run ID": "",
        "predictions_raw__test": "/home/cwatzenboeck/data/mlflow_cirpc_tmp/RA/data/190700519300523579/d377515686f1423b8f8f197a32c3eaf5/artifacts/predictions/test_/tmpj88ccxjy.npz",
    },
    "F_ERO_01": {
        #"Run ID": "",
        "Notes": "RMTANAEv2_SP_MH_FL_LR2_pp",
        "predictions_raw__test": "/home/cwatzenboeck/data/mlflow_cirpc_tmp/RA/data/973476768866622660/438c7be819a74f27b429129088e92c37/artifacts/predictions/test_/scores.npz",
    },
}


run = "H_ERO_01"
predictions_path = runs_to_examine[run]["predictions_raw__test"]
src = predictions_path

df = combine_predictions([src])
df["patientId_date_HF_LR"] = df["file_name"].apply(lambda x: "_".join(x.split("_")[:4]))
df["patientId_date_HF"] = df["file_name"].apply(lambda x: "_".join(x.split("_")[:3]))
df["patientId_date"] = df["file_name"].apply(lambda x: "_".join(x.split("_")[:2]))


# For now correct predictions... Class 6 can not occur anyhow...
df["labels"] = df["labels"].apply(lambda x: limit_treatment_number(x, limit=5, limit_treatment="over_limit_to_NA"))
df["preds"] = df["preds"].apply(lambda x: limit_treatment_number(x, limit=5, limit_treatment="over_limit_to_NA"))
df = df.dropna()
df_ERO_H  = df


#-------------------



run = "F_ERO_01"
predictions_path = runs_to_examine[run]["predictions_raw__test"]
src = predictions_path

df = combine_predictions([src])
df["patientId_date_HF_LR"] = df["file_name"].apply(lambda x: "_".join(x.split("_")[:4]))
df["patientId_date_HF"] = df["file_name"].apply(lambda x: "_".join(x.split("_")[:3]))
df["patientId_date"] = df["file_name"].apply(lambda x: "_".join(x.split("_")[:2]))


# For now correct predictions... Class 6 can not occur anyhow...
df["labels"] = df["labels"].apply(lambda x: limit_treatment_number(x, limit=5, limit_treatment="over_limit_to_NA"))
df["preds"] = df["preds"].apply(lambda x: limit_treatment_number(x, limit=5, limit_treatment="over_limit_to_NA"))
df = df.dropna()
df_ERO_F  = df





In [ ]:
#### INPUT:
FRACTION_REQUIRED_VALID_SCORES = 0.5

#### ERO H

In [ ]:


df = df_ERO_H
df_summed_H = sum_and_extrapolate_scores_df_ERO_H(df, fraction_required_valid_scores=FRACTION_REQUIRED_VALID_SCORES, limit_treatment_ED="E_D_mean")
df_delta_H = generate_score_differences(df_summed_H)

plot_SHS_sums(df_summed_H)
plt.show()

# plot_SHS_deltas(df_delta_H)
# plt.show()

### ERO F: 


In [ ]:


df = df_ERO_F
df_summed_F = sum_and_extrapolate_scores_df_ERO_F(df, fraction_required_valid_scores=FRACTION_REQUIRED_VALID_SCORES)


df_delta_F = generate_score_differences(df_summed_F)
plot_SHS_sums(df_summed_F)
plt.show()

### ERO F + H: 

In [ ]:

df_summed_H_F = (
    df_summed_H
    .set_index('patientId_date')
    .add(df_summed_F.set_index('patientId_date'), fill_value=None)
    .reset_index()
)


In [ ]:
plot_SHS_sums(df_summed_H_F.dropna(), name="ERO H+F")
plt.show()

In [ ]:
# df_summed_H_F.sort_values("preds_summed_extrapolated")

In [ ]:
# df_summed_H.sort_values("preds_summed_extrapolated")